In [1]:
import pandas as pd
import numpy as np
import spacy


In [2]:
import re

def nettoyer_phrase(phrase):
    phrase = phrase.strip()
    
   # uniformiser les tirets
    phrase = phrase.replace("–", "-").replace("—", "-")
    
    # enlever le tiret de dialogue en début
    phrase = re.sub(r"^\-\s*", "", phrase)
    
    # uniformiser les apostrophes
    phrase = phrase.replace("'", "’")
    
    # supprimer les espaces après une apostrophe élidée : s’ y -> s’y
    phrase = re.sub(r"([A-Za-zÀ-ÿ])’\s+([A-Za-zÀ-ÿ])", r"\1’\2", phrase)
    
    # espaces multiples
    phrase = re.sub(r"\s+", " ", phrase)
    
    return phrase.strip()

In [3]:
nlp = spacy.load("fr_core_news_lg")

In [4]:
# 1. Ouvrir le fichier et lire toutes les lignes d'un coup
with open("corpus_goncourt/1860_Charles Demailly_travail.txt", "r", encoding="utf-8") as f:
    lignes = f.readlines()


    

In [5]:
data = []

# 2. Boucler sur chaque ligne (qui est déjà une phrase)
for ligne in lignes:
    phrase_nettoyee = nettoyer_phrase(ligne) # Enlève les espaces et les \n invisibles
    
    if phrase_nettoyee: 
        data.append({
            'nom_fichier': "1860_Charles Demailly",
            'phrase': phrase_nettoyee
        })


df_final = pd.DataFrame(data)


print(f"Nombre de phrases chargées : {len(df_final)}")
df_final.head(10)

Nombre de phrases chargées : 1539


,nom_fichier,phrase
0,1860_Charles Demailly,Un article ?… Tu me demandes s’y a un article ...
1,1860_Charles Demailly,Hein ! la société ?…
2,1860_Charles Demailly,Tu ne la connais pas ? C’est pourtant une soci...
3,1860_Charles Demailly,Après ?
4,1860_Charles Demailly,"Après, après ? tu poses ta femme : une comédie..."
5,1860_Charles Demailly,Ceci était dit dans une grande pièce tendue d’...
6,1860_Charles Demailly,y avait cinq hommes dans cette pièce qui était...
7,1860_Charles Demailly,"L’un avait des cheveux blonds, le front court,..."
8,1860_Charles Demailly,L’autre était un jeune homme de trente-quatre ...
9,1860_Charles Demailly,"Des cheveux de femme, une bouche de femme, un ..."


In [6]:
df_final["phrase"] = df_final["phrase"].astype(str).str.strip()
df_final = df_final[df_final["phrase"] != ""].copy()

In [ ]:
def get_pos_with_punct(phrase):
    doc = nlp(phrase)
    return " ".join(token.text if token.pos_ == "PUNCT" else token.pos_ for token in doc) 
    # conserve la ponctuation telle quelle, et remplace les autres tokens par leur POS

df_final["pos"] = df_final["phrase"].apply(get_pos_with_punct)

In [8]:
df_final

,nom_fichier,phrase,pos
0,1860_Charles Demailly,Un article ?… Tu me demandes s’y a un article ...,DET NOUN ? … PRON PRON VERB PRON PRON VERB DET...
1,1860_Charles Demailly,Hein ! la société ?…,PROPN ! DET NOUN ? …
2,1860_Charles Demailly,Tu ne la connais pas ? C’est pourtant une soci...,PRON ADV DET NOUN ADV ? PRON AUX ADV DET NOUN ...
3,1860_Charles Demailly,Après ?,ADP ?
4,1860_Charles Demailly,"Après, après ? tu poses ta femme : une comédie...","ADP , ADP ? PRON VERB DET NOUN : DET NOUN ADJ ..."
...,...,...,...
1534,1860_Charles Demailly,Et ne répondit pas.,CCONJ ADV VERB ADV .
1535,1860_Charles Demailly,Nachette resta huit jours à Saint-Sauveur. fut...,PROPN VERB NUM NOUN ADP PROPN . AUX ADJ ADP AD...
1536,1860_Charles Demailly,"Bref, Nachette amusa Marthe, désarma Charles, ...","ADV , PROPN VERB PROPN , VERB PROPN , CCONJ , ..."
1537,1860_Charles Demailly,"Nachette parti, la campagne sembla à Marthe vi...","PROPN ADJ , DET NOUN VERB ADP PROPN NOUN ADP N..."


In [11]:
df_final.to_csv("corpus_goncourt/1860_Charles Demailly_motif_pos.csv", index=False, encoding="utf-8")

In [13]:
for token in nlp("La quatrième édition du livre de Burgard."):
    print(token.text, "|", token.pos_, "|", token.tag_, "|", token.morph)

La | DET | DET | Definite=Def|Gender=Fem|Number=Sing|PronType=Art
quatrième | ADJ | ADJ | Gender=Fem|NumType=Ord|Number=Sing
édition | NOUN | NOUN | Gender=Fem|Number=Sing
du | ADP | ADP | Definite=Def|Gender=Masc|Number=Sing|PronType=Art
livre | NOUN | NOUN | Number=Sing
de | ADP | ADP | 
Burgard | PROPN | PROPN | Gender=Masc|Number=Sing
. | PUNCT | PUNCT | 


In [14]:
import json
from pathlib import Path

motifs_path = Path("/Users/morganr/Champs_lexicaux_N-A/PyMotifs/src/motifs/data/fr_motifs.json")
with open(motifs_path, "r", encoding="utf-8") as f:
    motifs_dict = json.load(f)

def get_token_lower(token):
    return token.text.lower()

def get_token_lemma(token):
    return token.lemma_.lower()

def get_token_morph_set(token):
    morph_str = str(token.morph)
    return set(morph_str.split("|")) if morph_str else set()

def match_value(token_value, rule_value):
    if isinstance(rule_value, str):
        return token_value == rule_value
    if isinstance(rule_value, dict) and "IN" in rule_value:
        return token_value in rule_value["IN"]
    return False

def match_morph(token, morph_rule):
    token_morph = get_token_morph_set(token)
    if isinstance(morph_rule, dict) and "IS_SUPERSET" in morph_rule:
        required = set(morph_rule["IS_SUPERSET"])
        return required.issubset(token_morph)
    return False

def match_rule(token, rule):
    for key, rule_value in rule.items():
        if key == "POS":
            if not match_value(token.pos_, rule_value):
                return False
        elif key == "DEP":
            if not match_value(token.dep_, rule_value):
                return False
        elif key == "LOWER":
            if not match_value(get_token_lower(token), rule_value):
                return False
        elif key == "LEMMA":
            if not match_value(get_token_lemma(token), rule_value):
                return False
        elif key == "MORPH":
            if not match_morph(token, rule_value):
                return False
        else:
            return False
    return True

def rule_specificity(rule):
    score = 0
    for key, value in rule.items():
        score += 10
        if key in {"DEP", "MORPH"}:
            score += 10
        if key in {"LOWER", "LEMMA"}:
            score += 15
        if isinstance(value, dict) and "IN" in value:
            score += max(1, 10 - len(value["IN"]))
    return score

def build_ordered_patterns(motifs_dict):
    ordered = []
    for label, patterns in motifs_dict.items():
        for rule in patterns:
            ordered.append((label, rule, rule_specificity(rule)))
    ordered.sort(key=lambda x: x[2], reverse=True)
    return ordered

ordered_patterns = build_ordered_patterns(motifs_dict)

def token_to_motif(token, ordered_patterns, fallback="POS"):
    if token.is_punct:
        return token.text
    for label, rule, _score in ordered_patterns:
        if match_rule(token, rule):
            return label
    if fallback == "LEMMA":
        return token.lemma_
    elif fallback == "TEXT":
        return token.text
    return token.pos_

def phrase_to_motifs(phrase, nlp, ordered_patterns, fallback="POS"):
    doc = nlp(phrase)
    return " ".join(token_to_motif(token, ordered_patterns, fallback=fallback) for token in doc)

df_final["motif_pymotifs_like"] = df_final["phrase"].apply(
    lambda x: phrase_to_motifs(x, nlp, ordered_patterns, fallback="POS")
)

In [15]:
df_final

,nom_fichier,phrase,pos,motif_pymotifs_like
0,1860_Charles Demailly,Un article ?… Tu me demandes s’y a un article ...,DET NOUN ? … PRON PRON VERB PRON PRON VERB DET...,DET NC ? … tu me PRES PRON PRON avoir DET NCo ...
1,1860_Charles Demailly,Hein ! la société ?…,PROPN ! DET NOUN ? …,PROPN ! DET NC ? …
2,1860_Charles Demailly,Tu ne la connais pas ? C’est pourtant une soci...,PRON ADV DET NOUN ADV ? PRON AUX ADV DET NOUN ...,tu ne DET NC pas ? PRON être pourtant DET NC A...
3,1860_Charles Demailly,Après ?,ADP ?,après ?
4,1860_Charles Demailly,"Après, après ? tu poses ta femme : une comédie...","ADP , ADP ? PRON VERB DET NOUN : DET NOUN ADJ ...","après , après ? tu PRES DET NCo : DET NCo ADJ ..."
...,...,...,...,...
1534,1860_Charles Demailly,Et ne répondit pas.,CCONJ ADV VERB ADV .,CCONJ ne VPS pas .
1535,1860_Charles Demailly,Nachette resta huit jours à Saint-Sauveur. fut...,PROPN VERB NUM NOUN ADP PROPN . AUX ADJ ADP AD...,PROPN VPS NUM NCo ADP PROPN . être ADJ ADP tou...
1536,1860_Charles Demailly,"Bref, Nachette amusa Marthe, désarma Charles, ...","ADV , PROPN VERB PROPN , VERB PROPN , CCONJ , ...","ADV , PROPN VPS PROPN , PRES PROPN , CCONJ , A..."
1537,1860_Charles Demailly,"Nachette parti, la campagne sembla à Marthe vi...","PROPN ADJ , DET NOUN VERB ADP PROPN NOUN ADP N...","PROPN ADJ , DET NCs PRES ADP PROPN NC ADP NCmo..."
